# Chapter 8 Type Hints in Functions

## Group 1: Foundations of Gradual Typing & Duck vs. Nominal Typing

### Summary
* **Gradual Typing Principles (PEP 484):** Python's type system is strictly **optional**, **does not catch errors at runtime** (type hints are completely ignored by the Python runtime interpreter), and **does not improve execution performance**. Static type checkers (like Mypy) run separately to inspect source code.
* **Duck Typing vs. Nominal Typing:**
  * *Duck Typing:* Runtime-enforced based strictly on whether an object supports a given operation at call time, regardless of its declared class.
  * *Nominal Typing:* Statically enforced by type checkers based on explicit class names and inheritance hierarchies (`isinstance` / `issubclass`).
* **Static Protocols (`typing.Protocol`, PEP 544):** Enables *static duck typing* [. A class implements a `Protocol` simply by providing the required methods and signatures (e.g., `__lt__`), without needing to inherit from or register with the protocol class.

### Gotchas & Edge Cases
* **Runtime Blindness:** Passing an invalid type to an annotated parameter will not raise a runtime `TypeError` unless the function body itself hits an unsupported operation.
* **Ignored Unannotated Functions:** Mypy skips checking the body of functions that have no type annotations at all unless configured with strict flags like `--disallow-untyped-defs`.
* **Syntax Assignment Trap:** Writing `def fn(color=str)` sets the default parameter value to the `str` object rather than annotating its type (`def fn(color: str)`).

---

## Group 2: The Type System Hierarchy (`Any`, `object`, and Unions)

### Summary
* **`Any` vs. `object`:**
  * `object` sits at the top of the nominal class hierarchy. Annotating `x: object` accepts any value, but the type checker only permits operations supported by `object` itself (disallowing operations like multiplication or indexing).
  * `Any` is a wildcard at both the top and bottom of the static type hierarchy. Annotating `x: Any` accepts any value **and** assumes the value supports every possible operation, effectively turning off static checks for that variable.
* **Consistent-With Relationship:** Gradual typing replaces strict "subtype-of" with "consistent-with". `int` is declared consistent-with `float`, which is consistent-with `complex`.
* **Unions & `Optional`:**
  * `Optional[T]` is syntactic shorthand for `Union[T, None]`.
  * Python 3.10 introduced the infix `|` operator for union types (e.g., `str | None` or `int | float`).

### Gotchas & Edge Cases
* **`Optional` Does Not Make Arguments Optional:** Annotating `x: Optional[str]` tells the type checker the parameter accepts `str` or `None`, but it remains a mandatory argument at runtime unless assigned a default value (e.g., `x: str | None = None`).
* **Union Return Burden:** Returning `Union[A, B]` forces callers to execute `isinstance()` checks at runtime to determine which type was actually returned.

### Currency Check
* **BOOK vs CURRENT:** The book states that Unions and Optional can be written with the `|` operator introduced in Python 3.10; *What's new in Python 3.14* shows that `types.UnionType` and `typing.Union` are now unified aliases for each other, meaning `int | str` and `Union[int, str]` produce the exact same runtime type (What's new in Python 3.14, "typing" section).

---

## Group 3: Generic Collections, Tuples, Mappings, and ABCs

### Summary
* **Generic Collections (PEP 585):** Standard collections accept type parameters directly (e.g., `list[str]`, `dict[str, int]`).
* **Postel's Law for Function Annotations:**
  * *Be liberal in what you accept:* Annotate parameters with Abstract Base Classes from `collections.abc` (e.g., `Iterable[T]`, `Sequence[T]`, `Mapping[K, V]`) rather than concrete classes.
  * *Be conservative in what you send:* Annotate return values with concrete types (e.g., `list[str]`, `dict[str, set[str]]`) .
* **Tuple Annotations (Three Patterns):**
  1. *Record:* `tuple[str, float, str]` (fixed length, specific type per position).
  2. *Named Record:* `typing.NamedTuple` subclass.
  3. *Immutable Sequence:* `tuple[int, ...]` (arbitrary length, uniform type).

### Gotchas & Edge Cases
* **Missing ABC Methods:** `abc.Sequence` does not define `__mul__`, so a type checker will flag `x * 2` as an error if `x` is annotated as `Sequence`.
* **Numeric Tower Rejection:** The numeric ABCs from PEP 3141 (`numbers.Number`, `Real`, `Integral`) are explicitly rejected for static type checking by PEP 484. Use concrete types (`int`, `float`), explicit unions, or numeric static protocols (`SupportsFloat`) instead.

### Currency Check
* **BOOK vs CURRENT:** The book says redundant generic types in `typing` (like `typing.List`, `typing.Set`) are deprecated and scheduled for removal five years after Python 3.9 around Python 3.14; *What's New In Python 3.13 / 3.14* shows `typing.ByteString` is scheduled for removal in Python 3.17, while `typing.io` and `typing.re` were removed in Python 3.13 (What's New In Python 3.13 / 3.14).

---

## Group 4: Type Variables (`TypeVar`) & Generics

### Summary
* **`TypeVar`:** Used to parameterize generics so that parameter types are reflected in return types [40-42].
  * *Unrestricted:* `T = TypeVar('T')` [42].
  * *Restricted:* `NumberT = TypeVar('NumberT', float, Decimal, Fraction)` — limits `T` strictly to one of the choices listed [43].
  * *Bounded:* `HashableT = TypeVar('HashableT', bound=Hashable)` — allows `T` to be `Hashable` or any subtype of `Hashable` [44].
* **`AnyStr`:** A predefined restricted type variable defined as `TypeVar('AnyStr', bytes, str)` [45].

### Gotchas & Edge Cases
* **Unrestricted `TypeVar` Operations:** An unrestricted `TypeVar('T')` assumes `T` can be any type on first appearance, which will cause type checkers to reject operations like sorting (`<`) or hashing unless restricted or bounded [42, 46].

### Currency Check
* **BOOK vs CURRENT:** The book says `TypeVar` must be instantiated explicitly via `TypeVar('T')` and type aliases defined via `TypeAlias` in Python 3.10 (p. 281, 283) [47, 48]; *What's New In Python 3.12* shows PEP 695 introduced a native type parameter syntax (`def max[T](...):`) and the `type` statement (e.g., `type Point[T] = tuple[T, T]`), making explicit `TypeVar` calls optional for generic functions and aliases (What's New In Python 3.12, "PEP 695: Type Parameter Syntax") [49, 50].
* **BOOK vs CURRENT:** The book describes `AnyStr` as a predefined type variable (p. 286) [45]; *What's New In Python 3.13* shows `typing.AnyStr` is deprecated in Python 3.13, emits a runtime DeprecationWarning in 3.16, and will be removed in 3.18, recommending the new type parameter syntax instead (What's New In Python 3.13, "typing") [51].

---

## Group 5: Callables, Variance, `NoReturn`, and Signature Mechanics

### Summary
* **`Callable` Annotations:** Formatted as `Callable[[Param1, Param2], ReturnType]` or `Callable[..., ReturnType]` [52-54].
* **Callable Variance:**
  * *Covariant on return type:* A callback returning `int` is acceptable where a return of `float` is expected [55, 56].
  * *Contravariant on parameter types:* A callback accepting `complex` is acceptable where a parameter of `float` is expected [55, 56].
* **`NoReturn`:** Special return annotation for functions that never return normally (e.g., `sys.exit()` or functions that always raise exceptions) [57].
* **Positional-Only & Variadic Parameters:**
  * *Positional-only:* Indicated by `/` (Python 3.8+) or `__param` naming convention [58, 59].
  * *Variadic Arguments:* `*args: str` types each positional argument as `str` (inside the body, `args` is `tuple[str, ...]`); `**kwargs: float` types keyword values as `float` (inside the body, `kwargs` is `dict[str, float]`) [60].

### Gotchas & Edge Cases
* **Collection Invariance:** Unlike `Callable` return types, most generic collection types (like `list[float]`) are **invariant**—you cannot pass a `list[int]` where a `list[float]` is expected [61].
* **`Callable` Keyword Limitations:** `Callable` syntax cannot annotate optional or keyword arguments; use `Callable[..., ReturnType]` if function signatures vary [54].

### Currency Check
* **BOOK vs CURRENT:** The book describes stringifying annotations using `from __future__ import annotations` (PEP 563) (p. 272) [62]; *What's new in Python 3.14* shows PEP 649 & PEP 749 introduced deferred evaluation of annotations via annotate functions and `annotationlib`, and `from __future__ import annotations` is now deprecated (What's new in Python 3.14, "PEP 649 & PEP 749") [63-65].
* **BOOK vs CURRENT:** The book does not cover `typing.Self`, `TypedDict` `Unpack`, or `@typing.override`; *What's New In Python 3.11 / 3.12* introduced `typing.Self` (3.11, PEP 673), `Unpack[TypedDict]` for `**kwargs` typing (3.12, PEP 692), and `@typing.override` (3.12, PEP 698) (What's New In Python 3.11, 3.12) [66-69].


---

### 1. Unification of `types.UnionType` and `typing.Union` (Python 3.14)

#### Mechanism & Background
In Python 3.10 (PEP 604), the binary OR operator `|` was introduced to construct union types (e.g., `int | str`) [passage 430]. At runtime, evaluating `int | str` instantiated an object of type `types.UnionType` [passage 333]. On the other hand, using the traditional syntax `from typing import Union; Union[int, str]` instantiated an instance of a private internal class (`typing._UnionGenericAlias`) [passage 239, 334].

Because these two syntaxes instantiated distinct underlying runtime classes, runtime introspection tools had to maintain separate logic branches for `types.UnionType` and `typing._UnionGenericAlias` [passage 239, 334].

#### BOOK vs CURRENT: What Changed in Python 3.14
* **BOOK (Chapter 8, p. 270):** The book describes `Union[str, bytes]` from `typing` [passage 429, 431] and notes that Python 3.10 added `str | bytes` as complementary syntax [passage 430].
* **CURRENT (Python 3.14 Documentation):** In Python 3.14, `types.UnionType` and `typing.Union` were unified into aliases of each other [passage 333]. Both old-style (`Union[int, str]`) and new-style (`int | str`) unions now create instances of the **exact same runtime type** (`types.UnionType`) [passage 333].

#### Runtime Differences & Mechanics
1. **Unified String Representation:** Calling `repr(Union[int, str])` now outputs `"int | str"` rather than `"typing.Union[int, str]"` [passage 334].
2. **Cache Removal:** Old-style `Union[int, str]` objects are no longer cached at runtime [passage 334]. Previously, `Union[int, str] is Union[int, str]` evaluated to `True`. In Python 3.14, subscripting `Union` returns fresh objects every time [passage 334]. Always compare unions using equality (`==`), not identity (`is`) [passage 334].
3. **`isinstance` Support:** You can now perform runtime type checks directly against `typing.Union`:
   ```python
   isinstance(int | str, typing.Union)  # Evaluates to True in Python 3.14+
   ```
   *(Previously, passing `typing.Union` as the second argument to `isinstance` raised a `TypeError` [passage 334]).*
4. **Immutability Enforcement:** The `__args__` attribute on `Union` objects is now read-only, and setting dynamic attributes on `Union` instances is prohibited [passage 334].
5. **Deprecation Shim:** The private class `typing._UnionGenericAlias` is retained solely as a backward-compatibility shim and is scheduled for complete removal in Python 3.17 [passage 239, 334]. Introspection code should use `typing.get_origin()` and `typing.get_args()` instead [passage 239, 334].

---

### 2. Native Type Parameter Syntax (`PEP 695`) & Replacing `AnyStr`

#### The New Syntax (Python 3.12, PEP 695)
Before Python 3.12, creating generic functions, generic classes, or type aliases required importing and declaring explicit `TypeVar` objects (e.g., `T = TypeVar('T')`) [passage 81, 463, 467]. 

Python 3.12 introduced a compact **type parameter syntax** that attaches type parameters directly to functions, classes, and type aliases using brackets `[T]` [passage 81]:

```python
# Generic Function
def max[T](args: Iterable[T]) -> T: ...

# Generic Class
class list[T]:
    def __getitem__(self, index: int, /) -> T: ...

# Type Alias (using the new `type` statement)
type Point[T] = tuple[T, T]
``` 

#### Specifying Bounds and Constraints
Under PEP 695, bounds and constraints no longer require `TypeVar()` constructor arguments:
* **Bounded Type Variables:** Written with a colon `: BoundaryClass` (e.g., `[T: Hashable]`).
* **Constrained Type Variables:** Written with a tuple of permitted choices `: (Type1, Type2)` (e.g., `[T: (int, str)]`) 

```python
type HashableSequence[T: Hashable] = Sequence[T]     # Bounded
type IntOrStrSequence[T: (int, str)] = Sequence[T]  # Constrained
```

#### Replacing Deprecated `AnyStr`
* **BOOK (Chapter 8, p. 286):** The book presents `AnyStr` as a predefined restricted `TypeVar` constructed as `AnyStr = TypeVar('AnyStr', bytes, str)`.
* **CURRENT (Python 3.13 Documentation):** `typing.AnyStr` is **deprecated** in Python 3.13. It emits a runtime `DeprecationWarning` in Python 3.16 and will be removed entirely in Python 3.18.

**How to replace `AnyStr`:**
Replace `AnyStr` by writing generic functions with constrained type parameters `[S: (bytes, str)]`:

```python
# OLD (Deprecated):
from typing import AnyStr

def concat(a: AnyStr, b: AnyStr) -> AnyStr:
    return a + b

# NEW (Python 3.12+ / PEP 695):
def concat[S: (bytes, str)](a: S, b: S) -> S:
    return a + b
``` 

**How it works under the hood:** The type parameter `[S: (bytes, str)]` constrains `S` to be either strictly `bytes` or strictly `str`. If a caller passes `a` as `str` and `b` as `bytes`, the static type checker flags a mismatch, preserving the exact type-safety guarantees of `AnyStr` without requiring global `TypeVar` declarations.

---

### 3. Modern Type System Constructs: `Self`, `Unpack`, and `@override`

#### A. `typing.Self` (Python 3.11, PEP 673)
* **Problem Solved:** Annotating methods that return an instance of their enclosing class (e.g., fluent interface method chaining, or `classmethod` factory constructors) previously required declaring a verbose, bound `TypeVar` (e.g., `ST = TypeVar('ST', bound='MyClass')`) [passage 13].
* **How It Works:** `Self` dynamically represents the subtype of the class executing the method [passage 13, 14]. If a subclass inherits a method annotated as returning `Self`, a static type checker correctly infers that invoking the method on the subclass returns an instance of the subclass—not the parent class [passage 13, 14].

```python
from typing import Self

class MyLock:
    def __enter__(self) -> Self:
        self.lock()
        return self

class MyInt:
    @classmethod
    def fromhex(cls, s: str) -> Self:
        return cls(int(s, 16))
```

---

#### B. `Unpack` for `TypedDict` in `**kwargs` (Python 3.12, PEP 692)
* **Problem Solved:** Standard PEP 484 annotations for variadic keyword arguments (`**kwargs: str`) required **all** keyword argument values to share the same type (`str`). It was impossible to type individual keyword arguments with distinct types (e.g., `title: str` and `year: int`).
* **How It Works:** `Unpack[TypedDict]` allows a `TypedDict` class to specify the exact expected names and types of `**kwargs`.

```python
from typing import TypedDict, Unpack

class MovieParams(TypedDict):
    title: str
    year: int

def make_movie(**kwargs: Unpack[MovieParams]) -> None:
    ...

# Static type checker verifies keys and types at call time:
make_movie(title="Black Panther", year=2018)  # OK
make_movie(title="Star Wars", year="1977")    # ERROR: 'year' must be int
``` 

---

#### C. `@typing.override` Decorator (Python 3.12, PEP 698)
* **Problem Solved:** In object-oriented hierarchies, a developer might intend to override a superclass method, but accidentally misspell the method name (e.g., writing `get_colour()` instead of `get_color()`). Without explicit annotations, Python creates a new method on the subclass, leaving the parent method un-overridden without raising an error.
* **How It Works:** Decorating a subclass method with `@override` instructs static type checkers to verify that a method with the exact same name exists in an ancestor class. If no superclass method is found, the type checker raises an explicit error.

```python
from typing import override

class Base:
    def get_color(self) -> str:
        return "blue"

class GoodChild(Base):
    @override  # OK: Overrides Base.get_color
    def get_color(self) -> str:
        return "yellow"

class BadChild(Base):
    @override  # TYPE CHECKER ERROR: Base has no method named 'get_colour'!
    def get_colour(self) -> str:
        return "red"
``` 

---

## TODO
1. int is consistent with float and float is consistent with complex so int is also consistent with complex.
2. Union[int, float] is redundant because it can simply be using float to annotate the parameter and it will accept int as well.

In [ ]:
def show_count(count:int, singular: str, plural: str = '') -> str:
    if count == 1:
        return f'1 {singular}'
    count_str = str(count) if count else 'no'
    if not plural:
        plural = singular + 's'
    return f'{count_str} {plural}'

In [17]:
print(show_count(1, 'bird'))

1 bird


In [18]:
print(show_count(2, 'cat'))

2 cats


In [19]:
print(show_count(2, 'child', 'children'))

2 children


In [20]:
show_count(0, 'dog')

'no dogs'

In [28]:
def show_count2(count: int, singular: str, plural: str | None = None) -> str:
    if count == 1:
        return f'1 {singular}'
    count_str = str(count) if count else 'no'
    if not plural:
        plural = singular + 's'
    return f'{count_str} {plural}'

In [29]:
show_count2(2, 'cat')

'2 cats'

In [30]:
show_count2(2, 'child', 'children')

'2 children'

In [31]:
show_count2(0, 'dog')

'no dogs'

In [39]:
def f1(t: tuple[int, ...]) -> None:
    print(t)

f1(1), f1(())

1
()


(None, None)

In [35]:
f1(1)

1
